<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`:
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
%pip install -q beautifulsoup4 requests pandas lxml

In [2]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd
from io import StringIO

print("Libraries imported successfully.")

Libraries imported successfully.


and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()

    colunm_name = ' '.join(row.contents)

    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

# Fallback dataset containing the same web-scraped lab data.
fallback_csv_url = (
    "https://raw.githubusercontent.com/Roderic19/IBM-Applied-Data-Science-Capstone/"
    "main/spacex_web_scraped.csv"
)

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# TASK 1: Request the Falcon 9 Launch Wiki page from its URL

try:
    response = requests.get(static_url, headers=headers, timeout=30)
    response.raise_for_status()
    print("HTTP status:", response.status_code)
    print("Page downloaded successfully.")
except requests.RequestException as e:
    response = None
    print("Wikipedia request failed:", e)

HTTP status: 200
Page downloaded successfully.


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
# Create a BeautifulSoup object from the HTML response

if response is not None:
    soup = BeautifulSoup(response.text, "html.parser")
    print("BeautifulSoup object created successfully.")
else:
    soup = None
    print("BeautifulSoup could not be created because the page request failed.")

BeautifulSoup object created successfully.


Print the page title to verify if the `BeautifulSoup` object was created properly


In [7]:
# Print the page title

if soup is not None and soup.title is not None:
    print(soup.title.get_text(strip=True))
else:
    print("Page title is unavailable.")

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
# Find all tables on the Wiki page

if soup is not None:
    html_tables = soup.find_all("table", class_="wikitable")
    print("Number of tables found:", len(html_tables))
else:
    html_tables = []

Number of tables found: 13


Starting from the third table is our target table contains the actual launch records.


In [9]:
# The third table is the launch-record table in the course snapshot.
# If the page structure changes, select the first table containing the expected header.

if len(html_tables) > 2:
    first_launch_table = html_tables[2]
else:
    first_launch_table = None
    for table in html_tables:
        headers_found = [h.get_text(" ", strip=True) for h in table.find_all("th")]
        if any("Flight No." in h for h in headers_found):
            first_launch_table = table
            break

if first_launch_table is not None:
    print(first_launch_table.prettify()[:5000])
else:
    print("Target launch table was not found.")

<table class="wikitable plainrowheaders collapsible" id="mwA94" style="width: 100%;">
 <tbody id="mwA98">
  <tr id="mwA-A">
   <th id="mwA-E" scope="col">
    Flight No.
   </th>
   <th id="mwA-I" scope="col">
    Date and
    <br id="mwA-M"/>
    time (
    <a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwA-Q" rel="mw:WikiLink" title="Coordinated Universal Time">
     UTC
    </a>
    )
   </th>
   <th id="mwA-U" scope="col">
    <a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwA-Y" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">
     Version,
     <br id="mwA-c"/>
     Booster
    </a>
    <sup about="#mwt348" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"html":""},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"}},"i":0}}]}' id="cite_ref-booster_11-2" rel="dc:references" typeof="mw:Transclusion mw:

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []

if first_launch_table is not None:
    for th in first_launch_table.find_all("th"):
        name = extract_column_from_header(th)
        if name is not None and len(name) > 0:
            column_names.append(name)

print("Extracted column names:", column_names)

Extracted column names: ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
launch_dict = dict.fromkeys(column_names)

# Remove the irrelevant combined date/time column.
date_time_column = next(
    (c for c in launch_dict if c.startswith("Date and time")),
    None
)
if date_time_column is not None:
    del launch_dict[date_time_column]

# Initialize all output columns.
launch_dict["Flight No."] = []
launch_dict["Launch site"] = []
launch_dict["Payload"] = []
launch_dict["Payload mass"] = []
launch_dict["Orbit"] = []
launch_dict["Customer"] = []
launch_dict["Launch outcome"] = []
launch_dict["Version Booster"] = []
launch_dict["Booster landing"] = []
launch_dict["Date"] = []
launch_dict["Time"] = []

print("Output columns:", list(launch_dict.keys()))

Output columns: ['Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Version Booster', 'Booster landing', 'Date', 'Time']


Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
# TASK 3: Parse the launch tables into launch_dict

def cell_text(cell):
    if cell is None:
        return ""
    return unicodedata.normalize("NFKD", cell.get_text(" ", strip=True)).strip()

def cell_link_or_text(cell):
    if cell is None:
        return ""
    link = cell.find("a")
    if link is not None:
        value = link.get_text(" ", strip=True)
        if value:
            return value
    return cell_text(cell)

# Clear any previous values so the cell is safe to re-run.
for key in launch_dict:
    launch_dict[key] = []

extracted_row = 0

# The course snapshot contains several collapsible wikitable launch tables.
if soup is not None:
    launch_tables = soup.find_all("table", class_=lambda c: c and "wikitable" in c and "plainrowheaders" in c)
else:
    launch_tables = []

# Fall back to all wikitable tables if the class combination changed.
if not launch_tables:
    launch_tables = html_tables

for table_number, table in enumerate(launch_tables):
    for rows in table.find_all("tr"):
        if rows.th is None:
            continue

        flight_number = rows.th.get_text(" ", strip=True)
        # Remove footnote/reference text and keep a numeric flight number.
        flight_match = re.match(r"^(\d+)", flight_number)
        if not flight_match:
            continue

        row = rows.find_all("td")
        if len(row) < 9:
            continue

        extracted_row += 1

        datatimelist = date_time(row[0])
        date = datatimelist[0].strip(",") if len(datatimelist) > 0 else ""
        time = datatimelist[1] if len(datatimelist) > 1 else ""

        bv = booster_version(row[1]).strip()
        if not bv:
            bv = cell_link_or_text(row[1])

        launch_site = cell_link_or_text(row[2])
        payload = cell_link_or_text(row[3])
        payload_mass = get_mass(row[4])
        orbit = cell_link_or_text(row[5])
        customer = cell_text(row[6])
        launch_outcome = cell_text(row[7])
        booster_landing = landing_status(row[8])

        # Remove common footnote markers/noise.
        bv = re.sub(r"\s*\[[^\]]*\]", "", bv).strip()
        customer = re.sub(r"\s*\[[^\]]*\]", "", customer).strip()
        launch_outcome = re.sub(r"\s*\[[^\]]*\]", "", launch_outcome).strip()
        booster_landing = re.sub(r"\s*\[[^\]]*\]", "", booster_landing).strip()

        launch_dict["Flight No."].append(int(flight_number))
        launch_dict["Launch site"].append(launch_site)
        launch_dict["Payload"].append(payload)
        launch_dict["Payload mass"].append(payload_mass)
        launch_dict["Orbit"].append(orbit)
        launch_dict["Customer"].append(customer)
        launch_dict["Launch outcome"].append(launch_outcome)
        launch_dict["Version Booster"].append(bv)
        launch_dict["Booster landing"].append(booster_landing)
        launch_dict["Date"].append(date)
        launch_dict["Time"].append(time)

print("Rows extracted from Wikipedia:", extracted_row)

Rows extracted from Wikipedia: 121


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
# Create the dataframe

df = pd.DataFrame(launch_dict)

# If Wikipedia is temporarily unavailable or its HTML structure has changed,
# use the course-compatible web-scraped dataset as a fallback.
if df.empty or len(df) < 100:
    print("Wikipedia parsing did not produce the expected number of rows.")
    print("Loading the course-compatible fallback dataset...")
    try:
        fallback_df = pd.read_csv(fallback_csv_url)
        expected_columns = [
            "Flight No.", "Launch site", "Payload", "Payload mass", "Orbit",
            "Customer", "Launch outcome", "Version Booster",
            "Booster landing", "Date", "Time"
        ]
        if all(col in fallback_df.columns for col in expected_columns):
            df = fallback_df[expected_columns].copy()
        else:
            df = fallback_df.copy()
        print("Fallback dataset loaded:", df.shape)
    except Exception as e:
        print("Fallback dataset could not be loaded:", e)
        raise

# Standardize Flight No. and remove accidental duplicate rows.
if "Flight No." in df.columns:
    df["Flight No."] = pd.to_numeric(df["Flight No."], errors="coerce")
    df = df.dropna(subset=["Flight No."]).copy()
    df["Flight No."] = df["Flight No."].astype(int)
    df = df.drop_duplicates(subset=["Flight No."], keep="first").reset_index(drop=True)

print("Final dataframe shape:", df.shape)
display(df.head())

Final dataframe shape: (121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA ( COTS ) NRO,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA ( COTS ),Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA ( CRS ),Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA ( CRS ),Success,F9 v1.07B0007.1,No,1 March 2013,15:10


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab.

Following labs will be using a provided dataset to make each lab independent.


In [15]:
# Export the scraped dataset to CSV

output_file = "spacex_web_scraped.csv"
df.to_csv(output_file, index=False)
print(f"Saved {len(df)} rows to {output_file}")

Saved 121 rows to spacex_web_scraped.csv


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
